# Step 6: Live next-day forecast (technical indicators only)

Downloads recent S&P 500 prices, rebuilds the same technical features as step 2, and predicts whether the next trading day closes **Up or Down**.

**Why numerical-only:** the news dataset (`Combined_News_DJIA.csv`) ends in 2016 and there is no live headline source in this project, so sentiment cannot be computed for today. Step 5 showed that sentiment added almost nothing anyway.

**Important:** step 5 showed every model is at chance level on the test set (accuracy about 0.50). This output is a demonstration of the pipeline, not a trading signal.

## 6.1 Feature function (same formulas as `features.ipynb`)

In [ ]:
%pip install scikit-learn
import numpy as np
import pandas as pd
import yfinance as yf
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler

TECH_COLS = ["ret_1d", "ret_5d", "rsi_14", "macd", "macd_signal", "macd_hist", "vol_chg"]

def rsi(close, window=14):
    delta = close.diff()
    gain = delta.clip(lower=0)
    loss = -delta.clip(upper=0)
    avg_gain = gain.ewm(alpha=1 / window, adjust=False, min_periods=window).mean()
    avg_loss = loss.ewm(alpha=1 / window, adjust=False, min_periods=window).mean()
    rs = avg_gain / avg_loss
    return 100 - 100 / (1 + rs)

def make_features(prices):
    df = prices[["Date", "Close", "Volume"]].copy()
    df["ret_1d"] = df["Close"].pct_change()
    df["ret_5d"] = df["Close"].pct_change(5)
    df["rsi_14"] = rsi(df["Close"])
    ema12 = df["Close"].ewm(span=12, adjust=False).mean()
    ema26 = df["Close"].ewm(span=26, adjust=False).mean()
    macd = ema12 - ema26
    macd_signal = macd.ewm(span=9, adjust=False).mean()
    df["macd"] = macd / df["Close"] * 100
    df["macd_signal"] = macd_signal / df["Close"] * 100
    df["macd_hist"] = df["macd"] - df["macd_signal"]
    df["vol_chg"] = df["Volume"].pct_change().replace([np.inf, -np.inf], np.nan)
    return df

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 9.2/9.2 MB 3.4 MB/s eta 0:00:00a 0:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 35.3/35.3 MB 847.2 kB/s eta 0:00:0000:0100:02
   ━━━━━━╸━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1/6 [scipy]

## 6.2 Check it reproduces step 2

Applied to the 2008 to 2016 prices, it must match `data/features.csv` exactly.

In [2]:
old_prices = pd.read_csv("data/sp500_raw.csv", parse_dates=["Date"]).sort_values("Date").reset_index(drop=True)
saved = pd.read_csv("data/features.csv", parse_dates=["Date"])
rebuilt = make_features(old_prices).merge(saved[["Date"]], on="Date")
assert np.allclose(rebuilt[TECH_COLS].values, saved[TECH_COLS].values, atol=1e-9)
print("OK: feature function matches data/features.csv on", len(saved), "rows")

OK: feature function matches data/features.csv on 1979 rows


## 6.3 Train on all historical data

Logistic Regression with `C = 0.01` (the best numerical-only setting from step 5), fit on every row in `data/fused.csv` (train, validation and test together), with a scaler fit on the same rows.

In [3]:
hist = pd.read_csv("data/fused.csv", parse_dates=["Date"])
scaler = StandardScaler().fit(hist[TECH_COLS])
model = LogisticRegression(C=0.01, max_iter=1000).fit(scaler.transform(hist[TECH_COLS]), hist["target"])
print("trained on", len(hist), "days:", hist["Date"].min().date(), "to", hist["Date"].max().date())

trained on 1978 days: 2008-08-22 to 2016-06-30


## 6.4 Download recent prices

About 400 calendar days, enough warm-up for RSI and the exponential averages.

In [4]:
raw = yf.download("^GSPC", period="400d", interval="1d", auto_adjust=False, progress=False)
if isinstance(raw.columns, pd.MultiIndex):
    raw.columns = raw.columns.get_level_values(0)
recent = raw.reset_index()[["Date", "Open", "High", "Low", "Close", "Adj Close", "Volume"]]
recent["Date"] = pd.to_datetime(recent["Date"]).dt.tz_localize(None)
recent = recent.sort_values("Date").reset_index(drop=True)
print(len(recent), "rows,", recent["Date"].min().date(), "to", recent["Date"].max().date())
recent.tail(3)

400 rows, 2025-04-14 to 2026-11-13


Price,Date,Open,High,Low,Close,Adj Close,Volume
397,2026-11-11,7586.649902,7647.200195,7571.879883,7619.379883,7619.379883,4493200000
398,2026-11-12,7550.839844,7652.759766,7550.839844,7592.399902,7592.399902,4693040000
399,2026-11-13,7616.850098,7621.779785,7499.399902,7561.669922,7561.669922,4372990000


## 6.5 Predict the next trading day

In [5]:
feat = make_features(recent).dropna(subset=TECH_COLS)
latest = feat.iloc[[-1]]
p_up = model.predict_proba(scaler.transform(latest[TECH_COLS]))[0, 1]

print("latest close in the data:", latest["Date"].iloc[0].date(), f"| S&P 500 = {latest['Close'].iloc[0]:.2f}")
print("features used:")
print(latest[TECH_COLS].round(4).T.to_string(header=False))
print(f"\npredicted probability the NEXT trading day closes higher: {p_up:.3f}")
print("prediction:", "Up" if p_up >= 0.5 else "Down")
print("\nReminder: on the 2015-2016 test set this model type had about 50% accuracy (a coin flip).")

latest close in the data: 2026-11-13 | S&P 500 = 7561.67
features used:
Price               
ret_1d       -0.0040
ret_5d        0.0251
rsi_14       46.8103
macd         -0.5584
macd_signal  -0.5475
macd_hist    -0.0109
vol_chg      -0.0682

predicted probability the NEXT trading day closes higher: 0.561
prediction: Up

Reminder: on the 2015-2016 test set this model type had about 50% accuracy (a coin flip).


## 6.6 Is the model confident?

A probability close to 0.5 means the model has essentially no opinion. Compare with the spread of probabilities it gave on historical days.

In [6]:
hist_p = model.predict_proba(scaler.transform(hist[TECH_COLS]))[:, 1]
print(f"historical P(Up): min {hist_p.min():.3f}, median {np.median(hist_p):.3f}, max {hist_p.max():.3f}")
print(f"today's P(Up): {p_up:.3f}")

historical P(Up): min 0.348, median 0.540, max 0.686
today's P(Up): 0.561
